<a href="https://colab.research.google.com/github/Bhavana374/hallucination-detection-framework/blob/master/HaluEval_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/shashankjogur23/hallucination-detection-framework.git
%cd hallucination-detection-framework

!find . -maxdepth 4 -type f | sort


Cloning into 'hallucination-detection-framework'...
remote: Enumerating objects: 383, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 383 (delta 91), reused 377 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (383/383), 561.85 KiB | 18.73 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/hallucination-detection-framework
./api/__init__.py
./api/main.py
./api/routes/detect.py
./api/routes/__init__.py
./api/schemas/__init__.py
./api/schemas/schemas.py
./app/assets/README.md
./app/components/README.md
./app/main.py
./app/pages/README.md
./app/services/README.md
./configs/config.yaml
./configs/data.yaml
./configs/experiment.yaml
./configs/model.yaml
./configs/retrieval.yaml
./data/embeddings/.gitkeep
./data/interim/.gitkeep
./data/processed/.gitkeep
./data/raw/.gitkeep
./docs/architecture/README.md
./docs/CLOUD_TRAINING_GUIDE.md
./docs/meeting_notes/README.md
./docs/PROJECT_DOCUMENTATION.md
./docs/r

In [ ]:
import os
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ No GPU detected")

PyTorch: 2.11.0+cpu
CUDA available: False
⚠️ No GPU detected


In [ ]:
!ls -lah data/raw/
!find data/raw -maxdepth 3 -type f -print

total 12K
drwxr-xr-x 2 root root 4.0K Sep  3 05:48 .
drwxr-xr-x 6 root root 4.0K Sep  3 05:48 ..
-rw-r--r-- 1 root root   90 Sep  3 05:48 .gitkeep
data/raw/.gitkeep


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!find . -iname "*halueval*" -o -iname "*qa_samples*"

In [ ]:
!pwd


/content


In [ ]:
!git clone https://github.com/shashankjogur23/hallucination-detection-framework.git

Cloning into 'hallucination-detection-framework'...
remote: Enumerating objects: 383, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 383 (delta 91), reused 377 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (383/383), 561.85 KiB | 2.32 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [ ]:
%cd /content/hallucination-detection-framework

/content/hallucination-detection-framework


In [ ]:
!find data/raw -maxdepth 3 -type f -print

data/raw/.gitkeep


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("pminervini/HaluEval", "qa_samples")

print(dataset)

README.md:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

qa_samples/data-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.43MB            

qa_samples/data-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    data: Dataset({
        features: ['knowledge', 'question', 'answer', 'hallucination'],
        num_rows: 10000
    })
})


In [ ]:
import os
import json

os.makedirs("data/raw/halueval", exist_ok=True)

qa = dataset["data"]

output_path = "data/raw/halueval/qa_samples.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for row in qa:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_path)
print("Samples:", len(qa))

Saved: data/raw/halueval/qa_samples.jsonl
Samples: 10000


In [ ]:
import json
from collections import Counter

path = "data/raw/halueval/qa_samples.jsonl"

with open(path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

print("Total rows:", len(rows))
print("Fields:", list(rows[0].keys()))
print("Labels:", Counter(row["hallucination"] for row in rows))

print("Missing values:")
for field in ["knowledge", "question", "answer", "hallucination"]:
    print(f"  {field}:", sum(not row.get(field) for row in rows))

print("Unique answers:", len(set(row["answer"] for row in rows)))
print("Unique questions:", len(set(row["question"] for row in rows)))

Total rows: 10000
Fields: ['knowledge', 'question', 'answer', 'hallucination']
Labels: Counter({'yes': 5010, 'no': 4990})
Missing values:
  knowledge: 0
  question: 0
  answer: 0
  hallucination: 0
Unique answers: 9058
Unique questions: 10000


In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

labels = [1 if row["hallucination"] == "yes" else 0 for row in rows]

# 70% train, 30% temporary
train_rows, temp_rows = train_test_split(
    rows,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

temp_labels = [1 if row["hallucination"] == "yes" else 0 for row in temp_rows]

# Split remaining 30% equally -> 15% validation, 15% test
val_rows, test_rows = train_test_split(
    temp_rows,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

def stats(name, data):
    counts = Counter(row["hallucination"] for row in data)
    print(f"{name}: {len(data)} samples | {dict(counts)}")

stats("Train", train_rows)
stats("Validation", val_rows)
stats("Test", test_rows)

Train: 7000 samples | {'no': 3493, 'yes': 3507}
Validation: 1500 samples | {'yes': 752, 'no': 748}
Test: 1500 samples | {'no': 749, 'yes': 751}


In [ ]:
!python scripts/halueval_benchmark.py --help

usage: halueval_benchmark.py [-h] [--sanity-check] [--full]
                             [--sample-size SAMPLE_SIZE]
                             [--experiments [EXPERIMENTS ...]]
                             [--epochs EPOCHS] [--batch-size BATCH_SIZE]

HaluEval Benchmark Evaluation

options:
  -h, --help            show this help message and exit
  --sanity-check        Run sanity check on 200 samples only
  --full                Run full dataset (10,000 samples)
  --sample-size SAMPLE_SIZE
                        Subset sample size for CPU execution (default: 2000)
  --experiments [EXPERIMENTS ...]
                        Specific experiments to run (e.g., exp01 exp02 exp03
                        exp04 exp05 exp06 exp07 exp08)
  --epochs EPOCHS       Number of fine-tuning epochs for Transformer
                        experiments
  --batch-size BATCH_SIZE
                        Batch size for Transformer experiments


In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp04 --epochs 3 --batch-size 16

2026-09-03 06:09:18 | INFO     | halueval_benchmark:main:822 - Loading raw HaluEval data...
2026-09-03 06:09:18 | INFO     | halueval_benchmark:load_raw_data:61 - Loaded 10000 raw samples from /content/hallucination-detection-framework/data/raw/halueval/qa_samples.jsonl
2026-09-03 06:09:18 | INFO     | halueval_benchmark:main:825 - Validating data...
2026-09-03 06:09:18 | INFO     | halueval_benchmark:validate_data:123 - Validation: 9992 valid / 8 removed / 8 duplicates
2026-09-03 06:09:18 | INFO     | halueval_benchmark:main:832 - Validation report saved to /content/hallucination-detection-framework/data/processed/halueval/validation_report.json
2026-09-03 06:09:18 | INFO     | halueval_benchmark:main:840 - FULL DATASET MODE: Using all 10,000 samples
2026-09-03 06:09:18 | INFO     | halueval_benchmark:main:854 - Creating stratified train/val/test split (70/15/15)...
2026-09-03 06:09:18 | INFO     | halueval_benchmark:create_stratified_split:182 - Split: train=6996 (pos=3508, neg=3488)

In [ ]:
!sed -n '240,380p' scripts/halueval_benchmark.py


    t0 = time.time()
    model.fit(train_texts, train_labels)
    train_time = time.time() - t0

    t0 = time.time()
    test_probs = model.predict_proba(test_texts)
    test_preds = [1 if p >= 0.5 else 0 for p in test_probs]
    infer_time = time.time() - t0

    metrics = compute_classification_metrics(test_labels, test_preds, test_probs, train_time, infer_time)
    _save_experiment(output_dir, "exp02_tfidf_rf", {
        "model": "TF-IDF + Random Forest",
        "input_type": "claim_only",
        "max_features": 5000, "n_estimators": 100,
        "seed": RANDOM_SEED, "train_size": len(train_data), "test_size": len(test_data),
    }, metrics, test_data, test_labels, test_preds, test_probs)
    return metrics


def _finetune_transformer(model_obj, train_data, val_data, test_data, output_dir, exp_id, exp_name,
                          epochs=3, batch_size=16, lr=2e-5, warmup_ratio=0.1, max_grad_norm=1.0):
    """Shared fine-tuning logic for BERT/DeBERTa standalone experiments."""


In [ ]:
import torch
from src.models.deberta.deberta_classifier import DeBERTaClassifier

test_model = DeBERTaClassifier(
    model_name="microsoft/deberta-v3-base",
    num_labels=2,
    max_length=256
)

model = test_model.model
device = test_model.device

print("Device:", device)
print("Model dtype:", next(model.parameters()).dtype)

sample = rows[0]["answer"]

enc = test_model.tokenizer(
    sample,
    truncation=True,
    max_length=256,
    padding="max_length",
    return_tensors="pt"
)

input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Logits:", outputs.logits)
print("Contains NaN:", torch.isnan(outputs.logits).any().item())
print("Contains Inf:", torch.isinf(outputs.logits).any().item())

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

2026-09-03 06:15:57 | INFO     | deberta_classifier:__init__:64 - Loaded DeBERTa model 'microsoft/deberta-v3-base' on device 'cuda'
Device: cuda
Model dtype: torch.float16
Logits: tensor([[-0.0149,  0.0328]], device='cuda:0', dtype=torch.float16)
Contains NaN: False
Contains Inf: False


In [ ]:
!grep -n -E "float16|half\\(|\\.half\\(|fp16|dtype" src/models/deberta/deberta_classifier.py

39:        return np.array(probs, dtype=np.float32)
77:            return np.zeros((0, self.num_labels), dtype=np.float32)


In [ ]:
!grep -RniE "float16|fp16|half\\(|torch_dtype|dtype=" scripts src configs | head -100

scripts/halueval_benchmark.py:292:    train_dataset = TensorDataset(train_enc["input_ids"], train_enc["attention_mask"], torch.tensor(train_labels, dtype=torch.long))
scripts/halueval_benchmark.py:293:    val_dataset = TensorDataset(val_enc["input_ids"], val_enc["attention_mask"], torch.tensor(val_labels, dtype=torch.long))
src/retrieval/retrieval_engine.py:40:            return np.zeros((0, self.dim), dtype=np.float32)
src/retrieval/retrieval_engine.py:50:                pad = np.zeros((arr.shape[0], self.dim - arr.shape[1]), dtype=np.float32)
src/retrieval/retrieval_engine.py:56:            arr = np.zeros((len(texts), self.dim), dtype=np.float32)
src/retrieval/retrieval_engine.py:108:            return np.zeros((0, self.embedding_dim), dtype=np.float32)
src/features/feature_vectorizer.py:85:        vec = np.array([[f_dict[k] for k in self.feature_names]], dtype=np.float32)
src/features/feature_vectorizer.py:96:            return np.zeros((0, 0), dtype=np.float32), []
src/features/fea

In [ ]:
!sed -n '1,100p' src/models/deberta/deberta_classifier.py

"""
DeBERTa-v3 Fine-Tuned Classifier for Hallucination Detection (Exp 4 & Exp 6).
"""

import os
import time
from pathlib import Path
from typing import List, Dict, Any, Optional, Union

import numpy as np
from src.utils.logger import get_logger

logger = get_logger("deberta_classifier")

try:
    import torch
    import torch.nn as nn
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    logger.warning("PyTorch/Transformers unavailable. DeBERTa model running in fallback mode.")


class FallbackDeBERTaModel:
    """Fallback classifier when PyTorch/Transformers is not installed."""

    def __init__(self, num_labels: int = 2):
        self.num_labels = num_labels

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        probs = []
        for t in texts:
            val = (len(t) * 7 % 100) / 100.0
            if self.num_labels == 2:
                probs.append([1.0 - val, val])
   

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("Default CUDA dtype:", torch.get_default_dtype())

# Inspect the model parameters
dtypes = {}
for name, param in test_model.model.named_parameters():
    dtypes[str(param.dtype)] = dtypes.get(str(param.dtype), 0) + param.numel()

print("Parameter dtypes:")
for dtype, count in dtypes.items():
    print(f"  {dtype}: {count:,}")

print("\nClassifier:")
print(test_model.model.classifier)

CUDA: True
Default CUDA dtype: torch.float32
Parameter dtypes:
  torch.float16: 184,423,682

Classifier:
Linear(in_features=768, out_features=2, bias=True)


In [ ]:
!python - <<'PY'
from pathlib import Path

p = Path("src/models/deberta/deberta_classifier.py")
text = p.read_text()

old = '''self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)'''
new = '''self.model = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=num_labels,
                    torch_dtype=torch.float32
                )'''

if old not in text:
    raise RuntimeError("Expected model-loading line was not found.")

p.write_text(text.replace(old, new, 1))
print("✅ DeBERTa model loading updated to explicit FP32.")
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
✅ DeBERTa model loading updated to explicit FP32.


NameError: name 'PY' is not defined

In [ ]:
from src.models.deberta.deberta_classifier import DeBERTaClassifier

check_model = DeBERTaClassifier(
    model_name="microsoft/deberta-v3-base",
    num_labels=2,
    max_length=256,
    device="cuda"
)

dtypes = set(str(p.dtype) for p in check_model.model.parameters())

print("Device:", check_model.device)
print("Parameter dtypes:", dtypes)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

2026-09-03 06:20:25 | INFO     | deberta_classifier:__init__:64 - Loaded DeBERTa model 'microsoft/deberta-v3-base' on device 'cuda'
Device: cuda
Parameter dtypes: {'torch.float16'}


In [ ]:
from pathlib import Path

p = Path("src/models/deberta/deberta_classifier.py")
text = p.read_text()

old = """                self.model = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=num_labels,
                    torch_dtype=torch.float32
                )
                self.model.to(self.device)
"""

new = """                self.model = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=num_labels
                )
                self.model = self.model.float()
                self.model.to(self.device)
"""

if old in text:
    text = text.replace(old, new, 1)
    p.write_text(text)
    print("✅ Updated DeBERTa loading to force FP32 after loading.")
else:
    print("⚠️ Expected code pattern not found. Showing relevant section:")
    for i, line in enumerate(text.splitlines(), 1):
        if 50 <= i <= 70:
            print(f"{i}: {line}")

✅ Updated DeBERTa loading to force FP32 after loading.


In [ ]:
import importlib
import src.models.deberta.deberta_classifier as dc

importlib.reload(dc)

check_model = dc.DeBERTaClassifier(
    model_name="microsoft/deberta-v3-base",
    num_labels=2,
    max_length=256,
    device="cuda"
)

dtypes = set(str(p.dtype) for p in check_model.model.parameters())

print("Device:", check_model.device)
print("Parameter dtypes:", dtypes)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

2026-09-03 06:21:20 | INFO     | deberta_classifier:__init__:68 - Loaded DeBERTa model 'microsoft/deberta-v3-base' on device 'cuda'
Device: cuda
Parameter dtypes: {'torch.float32'}


In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp04 --epochs 3 --batch-size 16

2026-09-03 06:22:00 | INFO     | halueval_benchmark:main:822 - Loading raw HaluEval data...
2026-09-03 06:22:00 | INFO     | halueval_benchmark:load_raw_data:61 - Loaded 10000 raw samples from /content/hallucination-detection-framework/data/raw/halueval/qa_samples.jsonl
2026-09-03 06:22:00 | INFO     | halueval_benchmark:main:825 - Validating data...
2026-09-03 06:22:00 | INFO     | halueval_benchmark:validate_data:123 - Validation: 9992 valid / 8 removed / 8 duplicates
2026-09-03 06:22:00 | INFO     | halueval_benchmark:main:832 - Validation report saved to /content/hallucination-detection-framework/data/processed/halueval/validation_report.json
2026-09-03 06:22:00 | INFO     | halueval_benchmark:main:840 - FULL DATASET MODE: Using all 10,000 samples
2026-09-03 06:22:00 | INFO     | halueval_benchmark:main:854 - Creating stratified train/val/test split (70/15/15)...
2026-09-03 06:22:00 | INFO     | halueval_benchmark:create_stratified_split:182 - Split: train=6996 (pos=3508, neg=3488)

In [ ]:
from pathlib import Path
import json

result_dir = Path("results/halueval/full/exp04_deberta")

print("📁 Exp 4 directory exists:", result_dir.exists())
print("📄 Files saved:")

for p in sorted(result_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(result_dir))

print("\n📊 Experiment result:")
result_file = result_dir / "metrics.json"

if result_file.exists():
    print(json.dumps(json.loads(result_file.read_text()), indent=2))
else:
    print("⚠️ metrics.json not found")

📁 Exp 4 directory exists: True
📄 Files saved:
 - checkpoint/config.json
 - checkpoint/model.safetensors
 - checkpoint/tokenizer.json
 - checkpoint/tokenizer_config.json
 - classification_report.txt
 - config.json
 - confusion_matrix.png
 - metrics.json
 - predictions.csv
 - run_metadata.json

📊 Experiment result:
{
  "experiment_id": "exp04_deberta",
  "timestamp": "2026-09-03T06:30:15Z",
  "test_metrics": {
    "sample_count": 1498,
    "accuracy": 0.972,
    "precision": 0.985,
    "recall": 0.9587,
    "f1_score": 0.9717,
    "f1_macro": 0.972,
    "roc_auc": 0.9869,
    "pr_auc": 0.9907,
    "confusion_matrix": {
      "tp": 720,
      "tn": 736,
      "fp": 11,
      "fn": 31
    },
    "false_positive_rate": 0.0147,
    "false_negative_rate": 0.0413,
    "training_time_sec": 476.2885,
    "inference_time_sec": 7.8243,
    "latency_per_sample_ms": 5.223
  }
}


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)

model = model.float().cuda()

print("Device:", next(model.parameters()).device)
print("Parameter dtypes:", set(str(p.dtype) for p in model.parameters()))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda:0
Parameter dtypes: {'torch.float32'}


In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp03 --epochs 3 --batch-size 16

2026-09-03 06:35:14 | INFO     | halueval_benchmark:main:822 - Loading raw HaluEval data...
2026-09-03 06:35:14 | INFO     | halueval_benchmark:load_raw_data:61 - Loaded 10000 raw samples from /content/hallucination-detection-framework/data/raw/halueval/qa_samples.jsonl
2026-09-03 06:35:14 | INFO     | halueval_benchmark:main:825 - Validating data...
2026-09-03 06:35:14 | INFO     | halueval_benchmark:validate_data:123 - Validation: 9992 valid / 8 removed / 8 duplicates
2026-09-03 06:35:14 | INFO     | halueval_benchmark:main:832 - Validation report saved to /content/hallucination-detection-framework/data/processed/halueval/validation_report.json
2026-09-03 06:35:14 | INFO     | halueval_benchmark:main:840 - FULL DATASET MODE: Using all 10,000 samples
2026-09-03 06:35:14 | INFO     | halueval_benchmark:main:854 - Creating stratified train/val/test split (70/15/15)...
2026-09-03 06:35:14 | INFO     | halueval_benchmark:create_stratified_split:182 - Split: train=6996 (pos=3508, neg=3488)

In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp05 --epochs 5 --batch-size 16

2026-09-03 06:43:36 | INFO     | halueval_benchmark:main:822 - Loading raw HaluEval data...
2026-09-03 06:43:36 | INFO     | halueval_benchmark:load_raw_data:61 - Loaded 10000 raw samples from /content/hallucination-detection-framework/data/raw/halueval/qa_samples.jsonl
2026-09-03 06:43:36 | INFO     | halueval_benchmark:main:825 - Validating data...
2026-09-03 06:43:36 | INFO     | halueval_benchmark:validate_data:123 - Validation: 9992 valid / 8 removed / 8 duplicates
2026-09-03 06:43:36 | INFO     | halueval_benchmark:main:832 - Validation report saved to /content/hallucination-detection-framework/data/processed/halueval/validation_report.json
2026-09-03 06:43:36 | INFO     | halueval_benchmark:main:840 - FULL DATASET MODE: Using all 10,000 samples
2026-09-03 06:43:36 | INFO     | halueval_benchmark:main:854 - Creating stratified train/val/test split (70/15/15)...
2026-09-03 06:43:36 | INFO     | halueval_benchmark:create_stratified_split:182 - Split: train=6996 (pos=3508, neg=3488)

In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp06 --epochs 5 --batch-size 16

2026-09-03 06:46:46 | INFO     | halueval_benchmark:main:822 - Loading raw HaluEval data...
2026-09-03 06:46:46 | INFO     | halueval_benchmark:load_raw_data:61 - Loaded 10000 raw samples from /content/hallucination-detection-framework/data/raw/halueval/qa_samples.jsonl
2026-09-03 06:46:46 | INFO     | halueval_benchmark:main:825 - Validating data...
2026-09-03 06:46:46 | INFO     | halueval_benchmark:validate_data:123 - Validation: 9992 valid / 8 removed / 8 duplicates
2026-09-03 06:46:46 | INFO     | halueval_benchmark:main:832 - Validation report saved to /content/hallucination-detection-framework/data/processed/halueval/validation_report.json
2026-09-03 06:46:46 | INFO     | halueval_benchmark:main:840 - FULL DATASET MODE: Using all 10,000 samples
2026-09-03 06:46:46 | INFO     | halueval_benchmark:main:854 - Creating stratified train/val/test split (70/15/15)...
2026-09-03 06:46:46 | INFO     | halueval_benchmark:create_stratified_split:182 - Split: train=6996 (pos=3508, neg=3488)

In [ ]:
!grep -n -A120 -B30 "retrieval_score" scripts/halueval_benchmark.py

531-        "model": "Evidence-Grounded DeBERTa + NLI Fusion",
532-        "base_model": "microsoft/deberta-v3-base",
533-        "nli_model": "cross-encoder/nli-deberta-v3-small",
534-        "input_type": "claim+gold_evidence+nli_scores",
535-        "epochs": epochs,
536-        "batch_size": batch_size,
537-        "seed": RANDOM_SEED,
538-        "train_size": len(train_data),
539-        "val_size": len(val_data),
540-        "test_size": len(test_data),
541-    }
542-
543-    model.save(output_dir / "exp06_evidence_deberta" / "checkpoint")
544-    _save_experiment(output_dir, "exp06_evidence_deberta", config, metrics, test_data, test_labels, test_preds, test_probs)
545-    return metrics
546-
547-
548-def run_exp7_hybrid_rf(train_data, test_data, output_dir: Path) -> Dict:
549-    """Experiment 7: Hybrid Random Forest over 20-dimensional features."""
550-    from src.features.feature_vectorizer import FeatureVectorizer
551-    from src.models.hybrid.hybrid_classifier import Hybr

In [ ]:
!grep -n -A120 -B30 "retrieval_score" src/features/feature_vectorizer.py

35-        self.mean_ = np.mean(X, axis=0)
36-        self.scale_ = np.std(X, axis=0)
37-        self.scale_[self.scale_ == 0] = 1e-10
38-        return self
39-
40-    def transform(self, X: np.ndarray) -> np.ndarray:
41-        if self.mean_ is None or self.scale_ is None:
42-            return X
43-        return (X - self.mean_) / self.scale_
44-
45-    def fit_transform(self, X: np.ndarray) -> np.ndarray:
46-        return self.fit(X).transform(X)
47-
48-
49-class FeatureVectorizer:
50-    """
51-    Main Multi-Dimensional Feature Vectorizer engine for the Hybrid Model.
52-    """
53-
54-    def __init__(self, embedder=None, nli_model_name: str = "cross-encoder/nli-deberta-v3-small", excluded_features: Optional[List[str]] = None):
55-        self.nli_extractor = NLIFeatureExtractor(model_name=nli_model_name)
56-        self.semantic_extractor = SemanticFeatureExtractor(embedder=embedder)
57-        self.factual_extractor = FactualFeatureExtractor()
58-        self.linguistic_extra

In [ ]:
!grep -Rni -E "FAISS|SentenceTransformer|retriev|similarity" src scripts | head -100

src/pipeline/hallucination_detector.py:4:Executes response segmentation, claim extraction, evidence retrieval,
src/pipeline/hallucination_detector.py:13:from src.retrieval.retrieval_engine import EvidenceRetriever
src/pipeline/hallucination_detector.py:29:        retriever: Optional[EvidenceRetriever] = None,
src/pipeline/hallucination_detector.py:34:        retrieval_threshold: float = 0.30,
src/pipeline/hallucination_detector.py:37:        self.retriever = retriever or EvidenceRetriever()
src/pipeline/hallucination_detector.py:43:        self.retrieval_threshold = retrieval_threshold
src/pipeline/hallucination_detector.py:63:                "retrieval_score": 0.90 if lbl == 0 else 0.25
src/pipeline/hallucination_detector.py:73:    def determine_claim_verdict(self, hallucination_prob: float, retrieval_score: float, evidence_text: str) -> str:
src/pipeline/hallucination_detector.py:78:        - INSUFFICIENT_EVIDENCE: if evidence is absent or retrieval score is below threshold.
src/pipe

In [ ]:
!sed -n '1,210p' src/retrieval/retrieval_engine.py

"""
Evidence Retrieval Engine.

Encodes reference documents/context passages into dense vectors and retrieves
top-k relevant evidence sentences/passages for target claims using FAISSIndexer.
"""

import re
import numpy as np
from typing import List, Dict, Any, Optional
from src.retrieval.faiss_indexer import FAISSIndexer
from src.utils.logger import get_logger

logger = get_logger("retrieval_engine")

try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False
    logger.warning("sentence-transformers package not found. Using fallback lexical dense encoder.")

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False


class FallbackLexicalEncoder:
    """Fallback TF-IDF lexical dense encoder when sentence-transformers is unavailable."""

    def __init__(self, dim: int = 384):
        self.dim = dim
        self

In [ ]:
def run_exp7_hybrid_rf(train_data, test_data, output_dir: Path) -> Dict:
    """Experiment 7: Evidence-grounded Hybrid Random Forest."""

    from src.features.feature_vectorizer import FeatureVectorizer
    from src.models.hybrid.hybrid_classifier import HybridClassifier
    from src.retrieval.retrieval_engine import EvidenceRetriever
    from src.evaluation.metrics import compute_classification_metrics

    logger.info("=== EXP 7: Evidence-Grounded Hybrid Random Forest ===")

    train_labels = [1 if r["hallucination"] == "yes" else 0 for r in train_data]
    test_labels = [1 if r["hallucination"] == "yes" else 0 for r in test_data]

    # Initialize the project's real SentenceTransformer + FAISS retriever.
    retriever = EvidenceRetriever(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        embedding_dim=384,
        top_k=1,
        score_threshold=0.35,
    )

    def build_instances(data):
        instances = []

        for idx, r in enumerate(data):
            claim = r["answer"]
            knowledge = r["knowledge"]

            # Index this sample's provided knowledge as candidate evidence.
            retriever.indexer = retriever.indexer.__class__(
                embedding_dim=retriever.embedding_dim,
                metric="inner_product"
            )

            retriever.index_documents([
                {
                    "doc_id": f"sample_{idx}",
                    "text": knowledge
                }
            ])

            # Retrieve the most relevant evidence sentence.
            retrieved = retriever.retrieve_evidence(
                claim,
                top_k=1,
                claim_id=f"sample_{idx}"
            )

            if retrieved:
                best_evidence = retrieved[0]["text"]
                retrieval_score = float(
                    retrieved[0]["similarity_score"]
                )
            else:
                best_evidence = ""
                retrieval_score = 0.0

            instances.append({
                "claim": claim,
                "evidence": best_evidence,
                "retrieval_score": retrieval_score,
            })

        return instances

    logger.info("Extracting retrieved evidence and hybrid features...")
    train_instances = build_instances(train_data)
    test_instances = build_instances(test_data)

    vectorizer = FeatureVectorizer()

    model = HybridClassifier(
        classifier_type="random_forest",
        n_estimators=100,
        random_state=RANDOM_SEED,
        vectorizer=vectorizer
    )

    t0 = time.time()
    model.fit(train_instances, train_labels)
    train_time = time.time() - t0

    t0 = time.time()
    test_probs = model.predict_hallucination_score(test_instances)
    test_preds = [1 if p >= 0.5 else 0 for p in test_probs]
    infer_time = time.time() - t0

    metrics = compute_classification_metrics(
        test_labels,
        test_preds,
        test_probs,
        train_time,
        infer_time
    )

    # Get feature importances
    feat_importances = model.get_feature_importances()

    config = {
        "model": "Evidence-Grounded Hybrid Random Forest",
        "input_type": "claim+retrieved_evidence+multi_dimensional_features",
        "retrieval_model": "sentence-transformers/all-MiniLM-L6-v2",
        "retrieval_method": "FAISS inner-product cosine similarity",
        "top_k": 1,
        "score_threshold": 0.35,
        "evidence_source": "HaluEval provided knowledge",
        "n_estimators": 100,
        "n_features": len(model.vectorizer.feature_names),
        "feature_names": model.vectorizer.feature_names,
        "feature_importances": feat_importances,
        "seed": RANDOM_SEED,
        "train_size": len(train_data),
        "test_size": len(test_data),
    }

    # Save hybrid model checkpoint
    model_checkpoint_path = (
        PROJECT_ROOT
        / "models"
        / "checkpoints"
        / "hybrid"
        / "model.pkl"
    )
    model.save(model_checkpoint_path)

    _save_experiment(
        output_dir,
        "exp07_hybrid_rf",
        config,
        metrics,
        test_data,
        test_labels,
        test_preds,
        test_probs
    )

    return metrics

NameError: name 'Dict' is not defined

In [ ]:
!sed -n '1,160p' src/retrieval/faiss_indexer.py

"""
FAISS Vector Indexer for Evidence Retrieval.

Provides index building, top-k vector search, and persistence. Supports
fallback to NumPy-based cosine similarity matrix lookup if faiss is not installed.
"""

import os
import pickle
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from src.utils.logger import get_logger

logger = get_logger("faiss_indexer")

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    logger.warning("FAISS library not found. Falling back to NumPy matrix similarity search.")


class FAISSIndexer:
    """
    Vector index manager using FAISS (or NumPy fallback) for dense evidence retrieval.
    """

    def __init__(self, embedding_dim: int = 384, metric: str = "inner_product"):
        self.embedding_dim = embedding_dim
        self.metric = metric
        self.documents: List[Dict[str, Any]] = []
        self.embeddings: Optional[np.ndarray] = None
        self.index = None

        if HAS_FAISS:
     

In [ ]:
from pathlib import Path

path = Path("scripts/halueval_benchmark.py")

text = path.read_text()

old = '''    # Build instance dicts
    train_instances = [
        {"claim": r["answer"], "evidence": r["knowledge"], "retrieval_score": 0.90}
        for r in train_data
    ]
    test_instances = [
        {"claim": r["answer"], "evidence": r["knowledge"], "retrieval_score": 0.90}
        for r in test_data
    ]
'''

new = '''    # Build instance dicts using real SentenceTransformer + FAISS retrieval.
    from src.retrieval.retrieval_engine import EvidenceRetriever

    retriever = EvidenceRetriever(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        embedding_dim=384,
        top_k=1,
        score_threshold=0.35,
    )

    def build_instances(data):
        instances = []

        for idx, r in enumerate(data):
            claim = r["answer"]
            knowledge = r["knowledge"]

            # Each HaluEval sample provides its own candidate knowledge.
            # Retrieve the most relevant sentence from that knowledge.
            retriever.indexer = retriever.indexer.__class__(
                embedding_dim=retriever.embedding_dim,
                metric="inner_product",
            )

            retriever.index_documents([
                {
                    "doc_id": f"sample_{idx}",
                    "text": knowledge,
                }
            ])

            retrieved = retriever.retrieve_evidence(
                claim,
                top_k=1,
                claim_id=f"sample_{idx}",
            )

            if retrieved:
                best_evidence = retrieved[0]["text"]
                retrieval_score = float(
                    retrieved[0]["similarity_score"]
                )
            else:
                best_evidence = ""
                retrieval_score = 0.0

            instances.append({
                "claim": claim,
                "evidence": best_evidence,
                "retrieval_score": retrieval_score,
            })

        return instances

    logger.info("Extracting retrieved evidence and real retrieval scores...")
    train_instances = build_instances(train_data)
    test_instances = build_instances(test_data)
'''

if old not in text:
    raise RuntimeError("Expected Exp 7 code block was not found. Nothing was changed.")

path.write_text(text.replace(old, new))

print("✅ Exp 7 retrieval code updated successfully.")

✅ Exp 7 retrieval code updated successfully.


In [ ]:
!grep -n -A90 -B5 "run_exp7_hybrid_rf" scripts/halueval_benchmark.py

543-    model.save(output_dir / "exp06_evidence_deberta" / "checkpoint")
544-    _save_experiment(output_dir, "exp06_evidence_deberta", config, metrics, test_data, test_labels, test_preds, test_probs)
545-    return metrics
546-
547-
548:def run_exp7_hybrid_rf(train_data, test_data, output_dir: Path) -> Dict:
549-    """Experiment 7: Hybrid Random Forest over 20-dimensional features."""
550-    from src.features.feature_vectorizer import FeatureVectorizer
551-    from src.models.hybrid.hybrid_classifier import HybridClassifier
552-    from src.evaluation.metrics import compute_classification_metrics
553-
554-    logger.info("=== EXP 7: Hybrid Random Forest ===")
555-
556-    train_labels = [1 if r["hallucination"] == "yes" else 0 for r in train_data]
557-    test_labels = [1 if r["hallucination"] == "yes" else 0 for r in test_data]
558-
559-    # Build instance dicts using real SentenceTransformer + FAISS retrieval.
560-    from src.retrieval.retrieval_engine import EvidenceRetriever
5

In [ ]:
!python scripts/halueval_benchmark.py --full --experiments exp07

Streaming output truncated to the last 5000 lines.
2026-09-03 07:02:00 | INFO     | faiss_indexer:add_embeddings:76 - Added 1 vectors to index (Total: 1)
2026-09-03 07:02:00 | INFO     | retrieval_engine:index_documents:147 - Indexed 1 evidence sentence chunks from 1 documents.
2026-09-03 07:02:00 | INFO     | faiss_indexer:add_embeddings:76 - Added 1 vectors to index (Total: 1)
2026-09-03 07:02:00 | INFO     | retrieval_engine:index_documents:147 - Indexed 1 evidence sentence chunks from 1 documents.
2026-09-03 07:02:00 | INFO     | faiss_indexer:add_embeddings:76 - Added 1 vectors to index (Total: 1)
2026-09-03 07:02:00 | INFO     | retrieval_engine:index_documents:147 - Indexed 1 evidence sentence chunks from 1 documents.
2026-09-03 07:02:00 | INFO     | faiss_indexer:add_embeddings:76 - Added 3 vectors to index (Total: 3)
2026-09-03 07:02:00 | INFO     | retrieval_engine:index_documents:147 - Indexed 3 evidence sentence chunks from 1 documents.
2026-09-03 07:02:00 | INFO     | fais

In [ ]:
!cat results/halueval/full/exp07_hybrid_rf/metrics.json

{
  "experiment_id": "exp07_hybrid_rf",
  "timestamp": "2026-09-03T07:29:01Z",
  "test_metrics": {
    "sample_count": 1498,
    "accuracy": 0.972,
    "precision": 0.9746,
    "recall": 0.9694,
    "f1_score": 0.972,
    "f1_macro": 0.972,
    "roc_auc": 0.9942,
    "pr_auc": 0.9955,
    "confusion_matrix": {
      "tp": 728,
      "tn": 728,
      "fp": 19,
      "fn": 23
    },
    "false_positive_rate": 0.0254,
    "false_negative_rate": 0.0306,
    "training_time_sec": 1300.8657,
    "inference_time_sec": 278.6325,
    "latency_per_sample_ms": 186.003
  }
}